Apache Iceberg 的诞生，本质上是为了解决传统 Hive 表格式在云原生环境和现代数据工程需求下的架构性缺陷。它将数据湖从文件目录系统进化为真正的数据库表。

想象一下，以前使用 Hive 的时候: 

1. **并发即灾难:** 只要有人正在写入数据，其他人去查就会读到半截脏数据，或者因为文件锁冲突导致任务直接报错，根本没法实现真正的实时读写。
2. **改字段如走钢丝:** 想给表加个列或改个名？必须先停掉所有任务，手动同步元数据，稍有不慎就会导致元数据与底层文件不一致，整个报表链路瞬间断裂。
3. **查询全靠暴力扫描:** 引擎只知道数据在哪个文件夹，却不知文件里具体装了啥。哪怕只查一行数据，也得打开成千上万个文件全量遍历，速度慢到让人绝望。
4. **手误就是世界末日:** 一旦误删了数据或写错了批次，由于没有版本快照，根本无法一键回滚到昨天，只能靠痛苦的人工备份恢复，甚至直接数据丢失。

Apache Iceberg 是一种用于大型分析数据集的开放表格式。Iceberg使用高性能表格式，将表的概念附加到包括 Spark、Trino、PrestoDB、Flink、Hive 和 Impala 在内的计算引擎中，并且工作方式与 SQL 表完全相同。

其核心优势在于开放格式与事务能力的完美统一

1. **ACID 事务保障:** 在对象存储上实现了数据库级的强一致性，让多源并发写入、实时 Upsert 和精确快照隔离成为可能，彻底终结了数据脏读与竞态条件。
2. **模式演化自由:** 支持无缝的列添加、删除、重命名及类型升级，且向后兼容，让数据结构能随业务敏捷迭代，无需停机重构。
3. **计算存储彻底解耦:** 作为纯开放的表格式，它不被任何单一厂商锁定，允许 Spark、Flink、Trino、StarRocks 等引擎无缝互通同一份数据，真正实现了一次写入，随处计算。

---

## 核心概念
Apache Iceberg 是一种用于大型分析表的开放表格式。它将表视为一组文件，但通过独特的元数据层来管理这些文件，提供 ACID 事务、模式演进和时间旅行能力。

+ **Catalog (目录):** Iceberg 表的入口点。它存储表的元数据位置。常见的 Catalog 实现包括 Hadoop Catalog、Hive Metastore、Glue Catalog 和 REST Catalog。
+ **Snapshot (快照):** 表在某个时间点的完整状态。每次写入操作（Insert, Update, Delete, Merge）都会生成一个新的 Snapshot。Snapshot 是 Iceberg 实现时间旅行的基础。
+ **Manifest List (清单列表):** 每个 Snapshot 指向一个 Manifest List，它记录了该快照中包含的所有 Manifest File。
+ **Manifest File (清单文件):** 记录了一组数据文件的详细信息，包括文件路径、分区信息、列统计信息（Min/Max, Null Count）等。这是 Iceberg 实现高效过滤的关键。
+ **Data File (数据文件):** 实际存储数据的文件，支持 Parquet, ORC, Avro 等格式。
+ **Delete file(删除文件):** 用于编码通过位置或值删除的表中的行。

---

## 基础操作
### 创建 Catalog 与 表
首先配置 Catalog，然后创建一个带有分区策略的表。

```sql
-- 创建表: 按日期分区，并按用户 ID 分桶
CREATE TABLE dataspire_catalog.db_dev.user_events (
    user_id BIGINT,
    event_type STRING,
    event_time TIMESTAMP,
    amount DOUBLE
) USING iceberg
PARTITIONED BY (days(event_time), bucket(16, user_id));
```

### 写入与更新数据
Iceberg 支持标准的 INSERT, UPDATE, DELETE 和 MERGE INTO 操作。

```sql
-- 插入数据
INSERT INTO dataspire_catalog.db_dev.user_events 
VALUES (1001, 'click', timestamp('2023-10-01 10:00:00'), 0.0),
       (1002, 'purchase', timestamp('2023-10-01 10:05:00'), 99.5);

-- 更新数据 (基于主键或条件)
UPDATE dataspire_catalog.db_dev.user_events 
SET amount = 105.0 
WHERE user_id = 1002 AND event_type = 'purchase';

-- 删除数据
DELETE FROM dataspire_catalog.db_dev.user_events 
WHERE event_time < timestamp('2023-01-01 00:00:00');

-- Merge Into (Upsert 操作，常用于 CDC 场景)
MERGE INTO dataspire_catalog.db_dev.user_events AS target
USING source_table AS source
ON target.user_id = source.user_id AND target.event_time = source.event_time
WHEN MATCHED THEN UPDATE SET target.amount = source.amount
WHEN NOT MATCHED THEN INSERT (user_id, event_type, event_time, amount) 
VALUES (source.user_id, source.event_type, source.event_time, source.amount);
```

### 时间旅行
查询历史版本的数据，无需恢复备份。

```sql
-- 查询特定快照 ID 的数据
SELECT * FROM dataspire_catalog.db_dev.user_events VERSION AS OF 8957243;

-- 或者查询特定时间点的数据
SELECT * FROM dataspire_catalog.db_dev.user_events TIMESTAMP AS OF '2026-03-15 13:05:00';
```

---

## 底层机制解析
Apache Iceberg 采用列式存储，按列进行数据的编码、压缩和读取，能够实现极高的压缩比，同时减少大量非相关数据的扫描，从而更有效地利用 IO 和 CPU 资源。

Apache Iceberg 原生并不支持传统意义上的索引结构（如 B+ 树、Bitmap 或倒排索引），这是其架构设计的刻意选择，也是其高扩展性的秘诀所在。

Apache Iceberg 摒弃了传统数据库那样维护昂贵全局索引的思路，转而采用了一种更适应云存储和海量数据的元数据即索引策略。

<!-- 这是一张图片，ocr 内容为： -->
![](https://cdn.nlark.com/yuque/0/2026/png/45387277/1773549438295-f57ac4f1-c798-4c76-b3d5-07588ab45fce.png)

---

### Iceberg表格式
Iceberg 规范的V1，V2，V3已经完成，并被社区采纳。

V4 目前仍在积极开发中，尚未正式采用。

#### V1: 大型分析表
V1 定义了如何使用不可变文件格式（Parquet、Avro 和 ORC）来管理大型分析表。

#### V2: 行级删除
V2 的主要变化在于新增了删除文件以对现有数据文件中已删除的行进行编码的功能。此版本可用于在无需重写文件的情况下，对不可变数据文件删除或替换单个行。

#### V3: 扩展类型和功能
V3 扩展了数据类型和现有的元数据结构，以增加新的功能: 

+ 新的数据类型: nanosecond timestamp(tz), unknown, variant, geometry, geography
+ 列的默认值支持
+ 用于分区和排序的多参数转换
+ 行级血缘追踪
+ 二进制删除向量
+ 表加密密钥

---

### 乐观并发
Apache Iceberg 通过无锁的 MVCC（多版本并发控制）机制，彻底打破了传统数据仓库的写锁瓶颈。其核心在于利用底层存储的原子提交原语，将冲突检测后置到提交阶段，实现了真正的读写分离与高吞吐并行写入。

在遇到元数据冲突时，它能智能地基于最新快照重算并提交，对上层应用完全透明。这不仅保证了串行化隔离级别的强一致性，更让海量数据的高并发写入变得高效、稳健且极具弹性。

### 序列号
作为每个快照（Snapshot）的全局唯一、单调递增标识，Apache Iceberg 中的序列号超越了简单的时间戳，为数据文件提供了确定性的版本排序。这一机制让 Iceberg 能够精准地执行增量扫描（只读取特定序列号之后的新数据），并在 Merge-on-Read 场景中快速判断文件的新旧以解决写入冲突。

### 行级删除
Apache Iceberg 巧妙地将删除操作转化为高效的元数据事件，通过位置删除（Position Deletes）和等值删除（Equality Deletes）两种模式，在不重写底层海量数据文件的前提下，实现了毫秒级的逻辑删除可见性。这一机制完美解耦了删除标记与物理文件，既避免了传统 `UPDATE/DELETE` 带来的昂贵文件重写开销，又确保了查询引擎能即时过滤掉已删数据。

### 隐藏分区
在 Hive 中用户需显式处理分区值，而 Iceberg 允许用户查询时不需要知道具体的分区转换逻辑。

+ 用户写入 `event_time` (Timestamp)。
+ 物理存储按 `days(event_time)` 分区。
+ 查询时只需写 `WHERE event_time >= '...'`，Iceberg 自动将其转换为对底层分区文件的过滤。

---

### 高效扫描
它彻底摒弃了传统 Hive 式先列出所有文件再过滤的低效模式，转而采用基于元数据的谓词下推策略。查询引擎无需触碰任何数据文件，仅通过读取轻量级的Manifest List和Manifest File，利用内置的列统计信息（Min/Max、Null Count）和分区演化历史，就能在内存中瞬间剪枝掉 99% 无关的数据文件。

强烈建议在写入时对数据进行排序（Sorting）或使用 Z-Ordering。这能极大提升 Manifest 中 Min/Max 统计信息的区分度，从而让“元数据索引”的剪枝效率达到极致，效果等同于构建了聚簇索引。

这种元数据优先的架构，将扫描计划的时间复杂度从  O(N) （文件总数）降低至 O(log⁡N)甚至常数级，确保了无论数据湖膨胀到何种规模，查询启动都能保持极低的延迟与极高的精准度，真正实现了只读所需，无视海量。

### 压缩与清理
+ **小文件问题:** 频繁的小批量写入会产生大量小文件。Iceberg 本身不自动合并小文件（这与 Paimon 的自动 Compaction 不同），通常需要定期运行 `REWRITE DATA FILES` 过程来合并小文件并优化布局。
+ **过期清理:** 随着 Snapshot 增多，元数据和旧数据文件会占用空间。需定期运行 `EXPIRE SNAPSHOTS` 来删除过期的快照及其关联的不再需要的文件。

```sql
-- 优化数据文件（合并小文件，排序）
CALL my_catalog.system.rewrite_data_files(table => 'db.user_events');

-- 清理过期的快照（保留最近 3 天的快照）
CALL my_catalog.system.expire_snapshots(table => 'db.user_events', retain_last => 3);
```